In [ ]:
# 1. Imports
import sys
import os

# Ensure we can import from src
if os.path.abspath('src') not in sys.path:
    sys.path.append(os.path.abspath('src'))
    
# Import Core Library
from fancy_core.decorators import step, expand, vectorize, summarize, reduce
from fancy_core.workflow import Workflow
from fancy_core.engine import Engine
from fancy_core.store import DatumStore, InMemoryStore
from fancy_core.cells import FancyCell

# Import Mock Functions
# (Adding root to path to find mock_functions)
root_dir = os.path.abspath(os.path.join('..')) # fallback
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

from mock_functions.mock_youtube_transcript_analysis import (
    fetch_channel_videos,
    extract_metadata,
    transcribe_video,
    segment_conversations,
    analyze_sentiment,
    generate_report
)

In [ ]:
# 2. Register & Decorate Functions
# We wrap the raw mock functions with our Fancy definitions

# 1 -> N (Generator)
fetch_channel_videos_step = step(expand(fetch_channel_videos))

# 1 -> 1 (Map)
extract_metadata_step = step(vectorize(extract_metadata))
transcribe_video_step = step(vectorize(transcribe_video))

# 1 -> N (FlatMap/Expand)
segment_conversations_step = step(expand(segment_conversations))

# 1 -> 1 (Map)
analyze_sentiment_step = step(vectorize(analyze_sentiment))

# N -> 1 (Reduce/Aggregate)
generate_report_step = step(summarize(generate_report))

print("Functions decorated and registered.")

# 3. Create Infrastructure
store = InMemoryStore()
wf = Workflow(name="YouTube Analysis Mock")
print(f"Workflow '{wf.name}' initialized.")

In [ ]:
# 4. Define the Workflow Graph

# Input: Just a string literal for now
channel_url = "https://youtube.com/@example_channel"

# Step 1: Get Videos
# Returns a StepWiring object containing the Step definition and Output Cell placeholder
w1 = fetch_channel_videos_step(channel_url=channel_url)

# Step 2: Parallel Fetch (Metadata & Transcripts)
# Note: We pass w1 (wiring), the decorator extracts the output ID automatically
w2_meta = extract_metadata_step(video_url=w1)
w2_trans = transcribe_video_step(video_url=w1)

# Step 3: Segment Conversations
# Takes transcripts and metadata. Both align 1-to-1 with videos.
w3_segments = segment_conversations_step(transcript=w2_trans, metadata=w2_meta)

# Step 4: Analyze (Sentiment)
w4_sentiment = analyze_sentiment_step(conversation=w3_segments)

# Step 5: Report (Aggregate)
w5_report = generate_report_step(sentiments=w4_sentiment)


# Manual Wiring (Since context manager isn't fully active in this version)
wf.steps = [
    w1.step,
    w2_meta.step,
    w2_trans.step,
    w3_segments.step,
    w4_sentiment.step,
    w5_report.step
]

print(f"Defined {len(wf.steps)} steps.")

In [ ]:
# 5. Stage / inspect
import json

print(f"Workflow: {wf.name}")
for i, s in enumerate(wf.steps):
    print(f"Step {i+1}: {s.function_slug} (ID: {str(s.step_id)[:8]})")
    print(f"   Inputs: {s.inputs}")
    print(f"   Outputs: {s.outputs}")

# Verify JSON serialization
# print(wf.to_json())

In [ ]:
# 6. Run the Workflow
engine = Engine(store)

print("Starting Execution...\n" + "="*40)
# Returns a dict of all cells (inputs and outputs)
results = engine.run(wf)
print("="*40 + "\nExecution Complete.")

In [ ]:
# 7. Check final output
final_cell_id = w5_report.outputs.id
final_cell = results[final_cell_id]

# Resolve the value from the store
final_value = store.resolve(final_cell)

print(f"Final Result Cell ID: {final_cell_id}")
print(f"Value: {final_value}")